# Laboratório — Estimação, likelihood, MLE e MAP

[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/02-statistics/notebooks/13-estimacao-likelihood-mle-map-laboratorio.ipynb)

Este laboratório reproduz os conceitos da Aula 13 com soluções analíticas, otimização numérica e simulação. Todas as fontes aleatórias usam seed fixa.

**Dependências:** Python 3.10+, NumPy, pandas, SciPy, Matplotlib e scikit-learn. No Colab, essas bibliotecas já vêm instaladas.

## 1. Preparação

Executaremos quatro estudos:

1. MLE Bernoulli por fórmula, grade e otimização;
2. prior Beta, posterior e MAP;
3. propriedades de estimadores em amostras repetidas;
4. MLE e MAP em regressão logística.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn
from IPython.display import display
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LogisticRegression

SEED = 20260907
np.set_printoptions(precision=6, suppress=True)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Seed:", SEED)

## 2. Bernoulli: construir e maximizar a likelihood

Os dados representam dez tentativas independentes, com oito sucessos. Para (0<p<1),

[
ell(p)=slog p+(n-s)log(1-p).
]

Vamos comparar três caminhos: solução analítica, busca em grade e otimização numérica da negative log-likelihood.

In [ ]:
dados = np.array([1, 1, 1, 0, 1, 1, 0, 1, 1, 1], dtype=int)
n = dados.size
s = int(dados.sum())

def log_likelihood_bernoulli(p, sucessos=s, total=n):
    if not 0 < p < 1:
        return -np.inf
    return sucessos * np.log(p) + (total - sucessos) * np.log1p(-p)

p_mle_analitico = s / n
grade = np.linspace(0.001, 0.999, 999)
loglik_grade = np.array([log_likelihood_bernoulli(p) for p in grade])
p_mle_grade = grade[np.argmax(loglik_grade)]

resultado = minimize_scalar(
    lambda p: -log_likelihood_bernoulli(p),
    bounds=(1e-12, 1 - 1e-12),
    method="bounded",
    options={"xatol": 1e-14},
)
p_mle_numerico = resultado.x

print(f"n={n}, sucessos={s}")
print(f"MLE analítico: {p_mle_analitico:.9f}")
print(f"MLE na grade:  {p_mle_grade:.9f}")
print(f"MLE numérico:  {p_mle_numerico:.9f}")
assert resultado.success
assert np.isclose(p_mle_analitico, 0.8)
assert abs(p_mle_numerico - p_mle_analitico) < 1e-7

In [ ]:
likelihood_relativa = np.exp(loglik_grade - loglik_grade.max())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(grade, likelihood_relativa, color="#2563eb")
axes[0].axvline(p_mle_analitico, color="#dc2626", ls="--", label="MLE = 0,8")
axes[0].set(xlabel="p", ylabel="likelihood relativa", title="Likelihood Bernoulli")
axes[0].legend()

axes[1].plot(grade, loglik_grade, color="#0f766e")
axes[1].axvline(p_mle_analitico, color="#dc2626", ls="--", label="MLE = 0,8")
axes[1].set(xlabel="p", ylabel="log-likelihood", title="Log-likelihood Bernoulli")
axes[1].legend()
plt.tight_layout()
plt.show()

### Por que usar log-likelihood?

Produtos de milhares de probabilidades menores que 1 podem sofrer *underflow* e virar zero em ponto flutuante. A soma dos logaritmos permanece finita e preserva o ponto de máximo.

In [ ]:
n_grande, s_grande, p_teste = 10_000, 8_000, 0.8
likelihood_direta = p_teste**s_grande * (1 - p_teste)**(n_grande - s_grande)
loglik_estavel = (
    s_grande * np.log(p_teste)
    + (n_grande - s_grande) * np.log1p(-p_teste)
)

print("Likelihood calculada diretamente:", likelihood_direta)
print(f"Log-likelihood estável: {loglik_estavel:.6f}")
assert likelihood_direta == 0.0
assert np.isfinite(loglik_estavel)

### Soluções de fronteira

Se todos os resultados forem sucessos, o MLE é (p=1); se todos forem fracassos, é (p=0). Isso não é erro da derivação, mas pode ser uma estimativa extrema e frágil em amostras pequenas.

In [ ]:
amostras_extremas = {
    "todos fracassos": np.zeros(5, dtype=int),
    "todos sucessos": np.ones(5, dtype=int),
}
for nome, amostra in amostras_extremas.items():
    print(f"{nome:17s} -> MLE = {amostra.mean():.1f}")

## 3. Prior Beta, posterior e MAP

Com prior (operatorname{Beta}(alpha,eta)) e (s) sucessos em (n) tentativas:

[
pmid Dsimoperatorname{Beta}(alpha+s,eta+n-s).
]

Se os dois parâmetros posteriores são maiores que 1, a moda — o MAP — é

[
widehat p_{	ext{MAP}}=
rac{alpha+s-1}{alpha+eta+n-2}.
]

In [ ]:
alpha, beta = 2, 2
alpha_post = alpha + s
beta_post = beta + n - s

p_map = (alpha_post - 1) / (alpha_post + beta_post - 2)
p_media_posterior = alpha_post / (alpha_post + beta_post)

print(f"MLE:              {p_mle_analitico:.9f}")
print(f"MAP Beta(2,2):    {p_map:.9f}")
print(f"Média posterior:  {p_media_posterior:.9f}")
print(f"Posterior: Beta({alpha_post}, {beta_post})")

assert np.isclose(p_map, 0.75)
assert np.isclose(p_media_posterior, 10 / 14)

In [ ]:
p = np.linspace(0.001, 0.999, 1000)
prior = beta_dist.pdf(p, alpha, beta)
posterior = beta_dist.pdf(p, alpha_post, beta_post)
likelihood_escalada = np.exp(
    np.array([log_likelihood_bernoulli(v) for v in p])
    - max(log_likelihood_bernoulli(v) for v in p)
)
likelihood_escalada *= posterior.max()

plt.figure(figsize=(9, 4.5))
plt.plot(p, prior, label="prior Beta(2,2)", lw=2)
plt.plot(p, likelihood_escalada, label="likelihood (reescalada)", lw=2)
plt.plot(p, posterior, label="posterior Beta(10,4)", lw=2)
plt.axvline(p_mle_analitico, color="#dc2626", ls="--", label="MLE")
plt.axvline(p_map, color="#7c3aed", ls=":", label="MAP")
plt.xlabel("p")
plt.ylabel("densidade / escala relativa")
plt.title("Prior × likelihood → posterior")
plt.legend()
plt.tight_layout()
plt.show()

### Sensibilidade à prior

A tabela abaixo mantém os mesmos dados e muda a prior. Uma prior mais concentrada em 0,5 exerce maior regularização. Ela deve ser escolhida e documentada antes de olhar o resultado que se deseja favorecer.

In [ ]:
priors = [(1, 1), (2, 2), (10, 10), (2, 8)]
linhas = []
for a, b in priors:
    a_post, b_post = a + s, b + n - s
    mapa = (a_post - 1) / (a_post + b_post - 2)
    media = a_post / (a_post + b_post)
    linhas.append({
        "prior": f"Beta({a},{b})",
        "média_prior": a / (a + b),
        "MAP": mapa,
        "média_posterior": media,
    })

tabela_priors = pd.DataFrame(linhas)
display(tabela_priors.round(4))

## 4. Distribuição amostral e consistência do MLE Bernoulli

Repetiremos o experimento 20 mil vezes para cada tamanho amostral. O estimador (widehat p) é não viesado e sua variância teórica é (p(1-p)/n). A variabilidade deve diminuir quando (n) cresce.

In [ ]:
rng = np.random.default_rng(SEED)
p_verdadeiro = 0.30
repeticoes = 20_000
tamanhos = [10, 50, 200, 1_000]

resumo_bernoulli = []
estimativas_por_n = {}
for tamanho in tamanhos:
    estimativas = rng.binomial(tamanho, p_verdadeiro, size=repeticoes) / tamanho
    estimativas_por_n[tamanho] = estimativas
    vies = estimativas.mean() - p_verdadeiro
    variancia = estimativas.var(ddof=0)
    mse = np.mean((estimativas - p_verdadeiro) ** 2)
    resumo_bernoulli.append({
        "n": tamanho,
        "média_empírica": estimativas.mean(),
        "viés_empírico": vies,
        "variância_empírica": variancia,
        "variância_teórica": p_verdadeiro * (1 - p_verdadeiro) / tamanho,
        "MSE": mse,
    })

df_bernoulli = pd.DataFrame(resumo_bernoulli)
display(df_bernoulli.round(8))

assert np.all(np.abs(df_bernoulli["viés_empírico"]) < 0.003)
assert np.allclose(
    df_bernoulli["variância_empírica"],
    df_bernoulli["variância_teórica"],
    rtol=0.06,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for tamanho in [10, 50, 200]:
    axes[0].hist(
        estimativas_por_n[tamanho],
        bins=30,
        density=True,
        alpha=0.45,
        label=f"n={tamanho}",
    )
axes[0].axvline(p_verdadeiro, color="black", ls="--", label="p verdadeiro")
axes[0].set(xlabel="estimativa", ylabel="densidade", title="Distribuição de p̂")
axes[0].legend()

rmse = np.sqrt(df_bernoulli["MSE"])
axes[1].loglog(tamanhos, rmse, "o-", label="RMSE empírico")
referencia = rmse.iloc[0] * np.sqrt(tamanhos[0] / np.array(tamanhos))
axes[1].loglog(tamanhos, referencia, "--", label="referência 1/√n")
axes[1].set(xlabel="n", ylabel="RMSE", title="Concentração com mais dados")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Variância normal: MLE pode ser viesado

Se a média também é desconhecida, o MLE da variância divide a soma de quadrados por (n). Seu valor esperado é ((n-1)sigma^2/n). Dividir por (n-1) corrige o viés, embora o estimador corrigido já não seja o MLE.

In [ ]:
rng = np.random.default_rng(SEED + 1)
mu_verdadeiro = 5.0
sigma2_verdadeira = 4.0
repeticoes_var = 20_000
resumo_var = []

for tamanho in [5, 20, 100]:
    amostras = rng.normal(
        mu_verdadeiro,
        np.sqrt(sigma2_verdadeira),
        size=(repeticoes_var, tamanho),
    )
    var_mle = amostras.var(axis=1, ddof=0)
    var_corrigida = amostras.var(axis=1, ddof=1)
    resumo_var.append({
        "n": tamanho,
        "E empírico MLE": var_mle.mean(),
        "E teórico MLE": (tamanho - 1) / tamanho * sigma2_verdadeira,
        "E empírico corrigido": var_corrigida.mean(),
        "variância verdadeira": sigma2_verdadeira,
    })

df_var = pd.DataFrame(resumo_var)
display(df_var.round(6))

assert np.allclose(
    df_var["E empírico MLE"],
    df_var["E teórico MLE"],
    rtol=0.015,
)
assert np.allclose(
    df_var["E empírico corrigido"],
    sigma2_verdadeira,
    rtol=0.015,
)

## 6. Regressão logística: NLL, MLE e MAP

Para uma observação binária, minimizar a *binary cross-entropy* equivale a minimizar a negative log-likelihood Bernoulli. Acrescentar uma prior gaussiana aos coeficientes leva a um termo quadrático no objetivo MAP.

O intercepto não será penalizado neste exemplo. 

In [ ]:
rng = np.random.default_rng(SEED + 2)
n_logistica = 1_000
x = rng.normal(size=n_logistica)
beta_verdadeiro = np.array([-0.4, 1.7])
X = np.column_stack([np.ones(n_logistica), x])
prob = expit(X @ beta_verdadeiro)
y = rng.binomial(1, prob)

def nll_logistica(beta):
    z = X @ beta
    # log(1 + exp(z)) - y*z, calculado de forma estável.
    return np.sum(np.logaddexp(0.0, z) - y * z)

resultado_mle = minimize(
    nll_logistica,
    x0=np.zeros(2),
    method="BFGS",
    options={"gtol": 1e-9, "maxiter": 2_000},
)
beta_mle = resultado_mle.x

# Conferência independente com scikit-learn, praticamente sem penalização.
modelo_sklearn = LogisticRegression(
    C=1e12,
    l1_ratio=0,
    solver="lbfgs",
    fit_intercept=True,
    max_iter=2_000,
    tol=1e-10,
)
modelo_sklearn.fit(x.reshape(-1, 1), y)
beta_sklearn = np.array([modelo_sklearn.intercept_[0], modelo_sklearn.coef_[0, 0]])

print("β verdadeiro:      ", beta_verdadeiro)
print("β MLE (SciPy):     ", beta_mle)
print("β MLE (sklearn):   ", beta_sklearn)
print("Diferença máxima:  ", np.max(np.abs(beta_mle - beta_sklearn)))

# BFGS pode reportar perda de precisão já muito próximo do ótimo;
# a concordância independente e o gradiente numérico validam a solução.
assert np.max(np.abs(beta_mle - beta_sklearn)) < 1e-5

In [ ]:
def estimar_map(tau):
    def objetivo(beta):
        penalidade = 0.5 * (beta[1] / tau) ** 2
        return nll_logistica(beta) + penalidade
    ajuste = minimize(
        objetivo,
        x0=beta_mle,
        method="BFGS",
        options={"gtol": 1e-9, "maxiter": 2_000},
    )
    return ajuste.x

taus = [0.25, 0.5, 1.0, 10.0]
linhas_map = []
for tau in taus:
    beta_map = estimar_map(tau)
    linhas_map.append({
        "tau": tau,
        "intercepto_MAP": beta_map[0],
        "coeficiente_MAP": beta_map[1],
        "|coeficiente|": abs(beta_map[1]),
    })

df_map = pd.DataFrame(linhas_map)
display(df_map.round(6))
print(f"Coeficiente MLE sem prior: {beta_mle[1]:.6f}")

assert df_map.loc[0, "|coeficiente|"] < df_map.loc[1, "|coeficiente|"]
assert df_map.loc[1, "|coeficiente|"] < df_map.loc[2, "|coeficiente|"]
assert df_map.loc[2, "|coeficiente|"] < df_map.loc[3, "|coeficiente|"]
assert abs(df_map.loc[3, "coeficiente_MAP"] - beta_mle[1]) < 0.01

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df_map["tau"], df_map["coeficiente_MAP"], "o-", label="MAP")
plt.axhline(beta_mle[1], color="#dc2626", ls="--", label="MLE")
plt.xscale("log")
plt.xlabel("τ da prior Normal(0, τ²)")
plt.ylabel("coeficiente de x")
plt.title("Prior mais concentrada produz maior contração")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Verificações finais e leitura dos resultados

O laboratório deve satisfazer simultaneamente:

- os três métodos Bernoulli concordam no MLE 0,8;
- o MAP Beta(2,2) é 0,75 e difere da média posterior;
- a variância de (widehat p) acompanha (p(1-p)/n);
- o MLE da variância normal exibe o viés teórico;
- SciPy e scikit-learn concordam na regressão logística;
- reduzir (	au) contrai o coeficiente MAP em direção a zero.

In [ ]:
checagens = {
    "MLE Bernoulli": np.isclose(p_mle_analitico, 0.8),
    "MAP Beta-Bernoulli": np.isclose(p_map, 0.75),
    "underflow demonstrado": likelihood_direta == 0.0,
    "variância de p̂ validada": np.allclose(
        df_bernoulli["variância_empírica"],
        df_bernoulli["variância_teórica"],
        rtol=0.06,
    ),
    "viés da variância validado": np.allclose(
        df_var["E empírico MLE"],
        df_var["E teórico MLE"],
        rtol=0.015,
    ),
    "MLE logístico validado": np.max(np.abs(beta_mle - beta_sklearn)) < 1e-5,
    "contração MAP validada": df_map.loc[0, "|coeficiente|"] < abs(beta_mle[1]),
}

for nome, passou in checagens.items():
    print(f"{'PASSOU' if passou else 'FALHOU'} — {nome}")
assert all(checagens.values())

## Desafios

1. Troque os dados Bernoulli por 2 sucessos em 3 tentativas e compare MLE, MAP e média posterior.
2. Use uma prior Beta(20,5). Explique a direção da mudança e por que a escolha precisaria de justificativa substantiva.
3. Repita a simulação da variância normal com média conhecida; verifique que dividir por (n) deixa de produzir o mesmo viés.
4. Aumente e diminua o tamanho do conjunto logístico. Observe como a influência da mesma prior muda.
5. Acrescente uma segunda feature quase igual à primeira. Examine a estabilidade dos coeficientes e relacione o resultado à identificabilidade.

Na Aula 14, essas estimativas pontuais serão acompanhadas por procedimentos de intervalo e análise de cobertura.